# DS317 - KHAI PHÁ DỮ LIỆU TRONG DOANH NGHIỆP (UIT)
## PIPELINE TIỀN XỬ LÝ DỮ LIỆU NOAA AIS 2025 TOÀN NĂM (TỪ TẢI THÔ ĐẾN CHIA 4 QUÝ CHỐNG LEAKAGE)
**BÀI TOÁN: DỰ BÁO THỜI ĐIỂM CẬP CẢNG CỦA TÀU BIỂN THAY THẾ ETA TỰ KHAI BÁO**  
**Nhóm thực hiện: Nhóm 03 (Lớp DS317.P11 / DS317.R11)**  
*Phân loại đồ án: Nâng cao (Cải tiến phương pháp ML/DL & Triển khai Demo Web App Output)*

---

### 💡 KIẾN TRÚC KHÔI PHỤC TOÀN DIỆN & CHỐNG DATA LEAKAGE 4 QUÝ

1. **Khôi phục trọn vẹn Pipeline từ dữ liệu thô NOAA Azure Blob + Self-Checkpoint Engine:**
   - **Chạy từ đầu (Cold-Start 365 ngày):** Tải tệp `ais-YYYY-MM-DD.csv.zst` từ `https://noaaocm.blob.core.windows.net/ais/csv2/csv2025/` -> **Pass 1** nhận diện buộc bến (`status == 5.0` & `sog < 0.5`, Anti-Overfragmentation) sinh `ata_catalog.parquet` -> **Pass 2** ghép `pd.merge_asof` sinh bảng thô gắn nhãn 18 cột (`noaa_ais_2025_raw_labeled.parquet`).
   - **Tự kế thừa Checkpoint (Auto-Resume):** Nếu trên `/kaggle/input/...` hoặc `/kaggle/working/` đã có sẵn `noaa_ais_2025_raw_labeled.parquet` hoặc cặp thô 18 cột `01_ban_tho_18_thuoc_tinh/{train,test}.parquet`, pipeline tự động nhận diện và bỏ qua bước tải 365 ngày `.csv.zst`.

2. **Phân chia 4 Quý ĐỘC LẬP TRƯỚC KHI thực hiện bất kỳ tiền xử lý nào (`Zero Quarter-Boundary Crossing`):**
   - $Q_1$: `[2025-01-01, 2025-04-01)` | $Q_2$: `[2025-04-01, 2025-07-01)`
   - $Q_3$: `[2025-07-01, 2025-10-01)` | $Q_4$: `[2025-10-01, 2026-01-01)`
   - Một hành trình `trip_id` chỉ được giữ lại khi **cả thời điểm khởi hành, toàn bộ quan trắc, và `ata_time` đều nằm trọn vẹn trong cùng một quý**. Mọi chuyến xuyên biên giới (`CROSSING_Q1_Q2`, `Q2_Q3`, `Q3_Q4`) bị loại bỏ triệt để, bảo đảm $\text{Trips}(Q_i) \cap \text{Trips}(Q_j) = \emptyset$.

3. **Chỉ Fit & Tiền xử lý sẵn cho Quý 1 (`Q1_PURE`), giữ nguyên `NaN` ở $Q_2, Q_3, Q_4$ (Phương án 2):**
   - **Fit tham số 100% trên `Q1_PURE`:** Trung vị 3 cấp (`mmsi -> vessel_type -> Cargo/Tanker -> global`) cho `length, width, draft` và `status_mode` được học hoàn toàn từ dữ liệu sạch của riêng $Q_1$ (`q1_imputation_artifacts.json` / `train_imputation_artifacts.json`).
   - **Bản 1 (`01_ban_tho_18_thuoc_tinh/q1..q4.parquet` - 18 cột):** Dữ liệu thô nguyên bản 4 quý (không lọc, không điền khuyết).
   - **Bản 2 (`02_ban_nen_tang_22_thuoc_tinh/q1..q4.parquet` - 22 cột) & Bản 3 (`03_ban_tien_xu_ly_27_thuoc_tinh/q1..q4.parquet` - 27 cột):**
     + **`q1.parquet`:** Đã lọc sạch nhãn huấn luyện (`Target_Hours > 0` & $v_{\text{implied}} \ge 1.62\text{ kn}$) và đã điền khuyết hoàn chỉnh từ tham số của `Q1_PURE` (`% NaN = 0%` trên thân vỏ/góc/status).
     + **`q2.parquet`, `q3.parquet`, `q4.parquet`:** Chỉ áp dụng bộ lọc hàng hải vật lý (`is_train=False`, **không lọc theo `Target_Hours`**) và tính đặc trưng hình học/lượng giác, **GIỮ NGUYÊN `NaN` gốc chưa điền khuyết** (`imputation_flag` mang giá trị `0` hoặc `-1`) để các thành viên tự chạy bước tiền xử lý động phù hợp trong từng kịch bản thực nghiệm.


In [ ]:
"""Cell 1: Khởi tạo thư viện, cấu hình tham số, 3 Schema và Checkpoint."""

# Standard library imports
from datetime import datetime, timedelta
import gc
import glob
import json
import os
import shutil
import sys
import time
from typing import Dict, Iterator, List, Optional, Set, Tuple
import urllib.request
import warnings

# Third-party imports
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Local application/library specific imports
# (None required for standalone execution)

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ===========================================================================
# CẤU HÌNH HỆ THỐNG, THỜI GIAN 4 QUÝ & NGƯỠNG HÀNG HẢI
# ===========================================================================
START_DATE = '2025-01-01'
END_DATE = '2025-12-31'
BASE_URL = 'https://noaaocm.blob.core.windows.net/ais/csv2/csv2025/'

CHUNK_SIZE = 1_500_000
DOWNLOAD_BUFFER_SIZE = 8 * 1024 * 1024
BATCH_SIZE_SCAN = 2_000_000
BATCH_SIZE_SPLIT = 1_000_000

# Ngưỡng hàng hải thay thế lọc Target_Hours <= 168.0
MAX_DIST_TANKER_NM = 1800.0
MAX_DIST_CARGO_NM = 2300.0
MIN_IMPLIED_SPEED_KNOTS = 1.62
PARQUET_COMPRESSION = 'zstd'

# Chế độ thực thi: 'FULL' | 'PASS1_ONLY' | 'PASS2_ONLY' | 'PATCH_3_DAYS'
RUN_MODE = 'FULL'
FORCE_REPROCESS_ALL = False
CLEANUP_TEMP_AFTER_MERGE = True

IS_KAGGLE = os.path.exists('/kaggle')
BASE_WORK_DIR = (
    '/kaggle/working' if IS_KAGGLE else './kaggle_work_fullyear'
)
RAW_TEMP_DIR = os.path.join(BASE_WORK_DIR, 'raw_temp')
CHECKPOINT_DIR = os.path.join(BASE_WORK_DIR, 'checkpoints')
PASS1_CHECKPOINT_DIR = os.path.join(CHECKPOINT_DIR, 'pass1_moored')
PASS2_PARTS_DIR = os.path.join(CHECKPOINT_DIR, 'pass2_parts')

CATALOG_PARQUET = os.path.join(BASE_WORK_DIR, 'ata_catalog.parquet')
CATALOG_RECON_PARQUET = os.path.join(
    BASE_WORK_DIR, 'ata_catalog_reconstructed.parquet'
)
RAW_LABELED_WORK_PARQUET = os.path.join(
    BASE_WORK_DIR, 'noaa_ais_2025_raw_labeled.parquet'
)
MANIFEST_QUARTER_PARQUET = os.path.join(
    BASE_WORK_DIR, 'noaa_ais_2025_trip_quarter_manifest.parquet'
)
IMPUTE_ARTIFACTS_JSON = os.path.join(
    BASE_WORK_DIR, 'train_imputation_artifacts.json'
)
Q1_ARTIFACTS_JSON = os.path.join(
    BASE_WORK_DIR, 'q1_imputation_artifacts.json'
)
Q1_MMSI_MEDIANS_PARQUET = os.path.join(
    BASE_WORK_DIR, 'q1_mmsi_medians.parquet'
)

DIR_RAW_SPLIT = os.path.join(BASE_WORK_DIR, '01_ban_tho_18_thuoc_tinh')
DIR_BASE_SPLIT = os.path.join(
    BASE_WORK_DIR, '02_ban_nen_tang_22_thuoc_tinh'
)
DIR_PREP_SPLIT = os.path.join(
    BASE_WORK_DIR, '03_ban_tien_xu_ly_27_thuoc_tinh'
)

for d in [
    BASE_WORK_DIR,
    RAW_TEMP_DIR,
    PASS1_CHECKPOINT_DIR,
    PASS2_PARTS_DIR,
    DIR_RAW_SPLIT,
    DIR_BASE_SPLIT,
    DIR_PREP_SPLIT,
]:
    os.makedirs(d, exist_ok=True)

# Ranh giới thời gian 4 quý chuẩn UTC
QUARTER_SPECS = {
    'Q1': {
        'code': 1,
        'start': '2025-01-01 00:00:00',
        'end': '2025-04-01 00:00:00',
        'file': 'q1.parquet',
        'is_train': True,
    },
    'Q2': {
        'code': 2,
        'start': '2025-04-01 00:00:00',
        'end': '2025-07-01 00:00:00',
        'file': 'q2.parquet',
        'is_train': False,
    },
    'Q3': {
        'code': 3,
        'start': '2025-07-01 00:00:00',
        'end': '2025-10-01 00:00:00',
        'file': 'q3.parquet',
        'is_train': False,
    },
    'Q4': {
        'code': 4,
        'start': '2025-10-01 00:00:00',
        'end': '2026-01-01 00:00:00',
        'file': 'q4.parquet',
        'is_train': False,
    },
}

# Danh sách ngày tải từ NOAA Azure Blob khi chạy cold-start
if RUN_MODE == 'PATCH_3_DAYS':
    date_list = ['2025-03-31', '2025-09-27', '2025-09-30']
else:
    _sdt = datetime.strptime(START_DATE, '%Y-%m-%d')
    _edt = datetime.strptime(END_DATE, '%Y-%m-%d')
    date_list = [
        (_sdt + timedelta(days=i)).strftime('%Y-%m-%d')
        for i in range((_edt - _sdt).days + 1)
    ]

# ===========================================================================
# 3 BỘ PYARROW SCHEMA CHUẨN ĐỒNG NHẤT (18, 22, 27 CỘT)
# ===========================================================================
SCHEMA_RAW_LABELED = pa.schema([
    ('mmsi', pa.int64()),
    ('trip_id', pa.string()),
    ('base_date_time', pa.timestamp('us')),
    ('longitude', pa.float32()),
    ('latitude', pa.float32()),
    ('sog', pa.float32()),
    ('cog', pa.float32()),
    ('heading', pa.float32()),
    ('status', pa.float32()),
    ('vessel_type', pa.float32()),
    ('length', pa.float32()),
    ('width', pa.float32()),
    ('draft', pa.float32()),
    ('cargo', pa.float32()),
    ('berth_lon', pa.float32()),
    ('berth_lat', pa.float32()),
    ('ata_time', pa.timestamp('us')),
    ('Target_Hours', pa.float32()),
])

SCHEMA_BASE = pa.schema([
    ('mmsi', pa.int64()),
    ('trip_id', pa.string()),
    ('base_date_time', pa.timestamp('us')),
    ('longitude', pa.float32()),
    ('latitude', pa.float32()),
    ('sog', pa.float32()),
    ('cog', pa.float32()),
    ('heading', pa.float32()),
    ('status', pa.float32()),
    ('vessel_type', pa.int32()),
    ('length', pa.float32()),
    ('width', pa.float32()),
    ('draft', pa.float32()),
    ('cargo', pa.float32()),
    ('berth_lon', pa.float32()),
    ('berth_lat', pa.float32()),
    ('ata_time', pa.timestamp('us')),
    ('Dist_To_Berth_NM', pa.float32()),
    ('Area_Proxy', pa.float32()),
    ('Volume_Proxy', pa.float32()),
    ('Is_Tanker', pa.int32()),
    ('Target_Hours', pa.float32()),
])

SCHEMA_PREPROCESSED = pa.schema([
    ('mmsi', pa.int64()),
    ('trip_id', pa.string()),
    ('base_date_time', pa.timestamp('us')),
    ('longitude', pa.float32()),
    ('latitude', pa.float32()),
    ('sog', pa.float32()),
    ('cog', pa.float32()),
    ('heading', pa.float32()),
    ('cog_sin', pa.float32()),
    ('cog_cos', pa.float32()),
    ('heading_sin', pa.float32()),
    ('heading_cos', pa.float32()),
    ('status', pa.float32()),
    ('vessel_type', pa.int32()),
    ('length', pa.float32()),
    ('width', pa.float32()),
    ('draft', pa.float32()),
    ('cargo', pa.float32()),
    ('berth_lon', pa.float32()),
    ('berth_lat', pa.float32()),
    ('ata_time', pa.timestamp('us')),
    ('Dist_To_Berth_NM', pa.float32()),
    ('Area_Proxy', pa.float32()),
    ('Volume_Proxy', pa.float32()),
    ('Is_Tanker', pa.int32()),
    ('imputation_flag', pa.int32()),
    ('Target_Hours', pa.float32()),
])

COLS_RAW = [f.name for f in SCHEMA_RAW_LABELED]
COLS_BASE = [f.name for f in SCHEMA_BASE]
COLS_PREP = [f.name for f in SCHEMA_PREPROCESSED]


# ===========================================================================
# HỆ THỐNG TỰ ĐỘNG PHÁT HIỆN CHECKPOINT ĐA TẦNG (SELF-CHECKPOINT ENGINE)
# ===========================================================================
def resolve_input_dirs() -> List[str]:
    """Liệt kê các thư mục đầu vào hợp lệ theo thứ tự ưu tiên."""
    dirs: List[str] = []
    env_dir = os.environ.get('AIS_INPUT_DIR')
    if env_dir and os.path.isdir(env_dir):
        dirs.append(env_dir)

    candidates = [
        '/kaggle/input/noaa-ais-2025-full-year-tin-x-l-nn-tng',
        '/kaggle/input/datasets/kienvo134/'
        'noaa-ais-2025-full-year-tin-x-l-nn-tng',
        BASE_WORK_DIR,
    ]
    for c in candidates:
        if os.path.isdir(c) and c not in dirs:
            dirs.append(c)

    if os.path.isdir('/kaggle/input'):
        dirs.append('/kaggle/input')
    return dirs


def find_existing_file(filename: str) -> Optional[str]:
    """Tìm tệp checkpoint theo tên trong working hoặc input."""
    work_candidate = os.path.join(BASE_WORK_DIR, filename)
    if os.path.isfile(work_candidate) and os.path.getsize(work_candidate) > 0:
        return work_candidate

    for base_d in resolve_input_dirs():
        direct = os.path.join(base_d, filename)
        if os.path.isfile(direct) and os.path.getsize(direct) > 0:
            return direct
        matches = glob.glob(
            os.path.join(base_d, '**', filename), recursive=True
        )
        for m in matches:
            if os.path.isfile(m) and os.path.getsize(m) > 0:
                return m
    return None


def find_raw_18_sources() -> Tuple[str, List[str]]:
    """Dò tìm nguồn dữ liệu thô 18 cột sẵn có (đơn lẻ hoặc cặp split).

    Returns:
        Tuple[str, List[str]]: (mode, list_paths) trong đó mode là
            'SINGLE_RAW', 'SPLIT_PAIR', hoặc 'NONE'.
    """
    single = find_existing_file('noaa_ais_2025_raw_labeled.parquet')
    if single is not None and os.path.getsize(single) > 1024:
        return 'SINGLE_RAW', [single]

    for base_d in resolve_input_dirs():
        sub_dir = os.path.join(base_d, '01_ban_tho_18_thuoc_tinh')
        tr_p = os.path.join(sub_dir, 'train.parquet')
        te_p = os.path.join(sub_dir, 'test.parquet')
        if os.path.isfile(tr_p) and os.path.isfile(te_p):
            return 'SPLIT_PAIR', [tr_p, te_p]
        tr_matches = glob.glob(
            os.path.join(base_d, '**', '01_ban_tho_18_thuoc_tinh',
                         'train.parquet'),
            recursive=True,
        )
        te_matches = glob.glob(
            os.path.join(base_d, '**', '01_ban_tho_18_thuoc_tinh',
                         'test.parquet'),
            recursive=True,
        )
        if tr_matches and te_matches:
            return 'SPLIT_PAIR', [tr_matches[0], te_matches[0]]

    return 'NONE', []


def get_available_pass1_checkpoints() -> Dict[str, str]:
    """Thu thập danh sách checkpoint moored_*.parquet của Pass 1."""
    avail: Dict[str, str] = {}
    for f in glob.glob(os.path.join(PASS1_CHECKPOINT_DIR, 'moored_*.parquet')):
        if os.path.getsize(f) > 100:
            d_str = os.path.basename(f)[7:-8]
            avail[d_str] = f
    if os.path.isdir('/kaggle/input'):
        for f in glob.glob(
            '/kaggle/input/**/moored_*.parquet', recursive=True
        ):
            if os.path.getsize(f) > 100:
                d_str = os.path.basename(f)[7:-8]
                if d_str not in avail:
                    avail[d_str] = f
    return avail


def get_available_pass2_parts() -> Dict[str, str]:
    """Thu thập danh sách part_raw_*.parquet của Pass 2."""
    avail: Dict[str, str] = {}
    for f in glob.glob(os.path.join(PASS2_PARTS_DIR, 'part_raw_*.parquet')):
        if os.path.getsize(f) > 100:
            d_str = os.path.basename(f)[9:-8]
            avail[d_str] = f
    if os.path.isdir('/kaggle/input'):
        for f in glob.glob(
            '/kaggle/input/**/part_raw_*.parquet', recursive=True
        ):
            if os.path.getsize(f) > 100:
                d_str = os.path.basename(f)[9:-8]
                if d_str not in avail:
                    avail[d_str] = f
    return avail


RAW_SOURCE_MODE, RAW_SOURCE_PATHS = find_raw_18_sources()
SELF_CATALOG_FILE = find_existing_file('ata_catalog.parquet')
SELF_CATALOG_RECON = find_existing_file('ata_catalog_reconstructed.parquet')
SELF_SPLIT_MANIFEST = find_existing_file(
    'noaa_ais_2025_trip_split_manifest.parquet'
)
SELF_PASS1_PARTS = get_available_pass1_checkpoints()
SELF_PASS2_PARTS = get_available_pass2_parts()

d_tot, _, d_free = shutil.disk_usage(BASE_WORK_DIR)
print('=' * 75)
print(f'Python: {sys.version.split()[0]} | Pandas: {pd.__version__} | '
      f'PyArrow: {pa.__version__}')
print(f'Môi trường       : {"Kaggle Kernel" if IS_KAGGLE else "Local"}')
print(f'RUN_MODE         : {RUN_MODE} ({len(date_list)} ngày) | '
      f'FORCE_REPROCESS_ALL={FORCE_REPROCESS_ALL}')
print(f'Đĩa working      : Trống {d_free / (1024 ** 3):.2f} GiB / '
      f'{d_tot / (1024 ** 3):.2f} GiB')
print(f'Nguồn thô 18 cột : {RAW_SOURCE_MODE} -> {RAW_SOURCE_PATHS}')
print(f'ata_catalog      : {SELF_CATALOG_FILE}')
print(f'Manifest gốc     : {SELF_SPLIT_MANIFEST}')
print(f'Checkpoint P1/P2 : {len(SELF_PASS1_PARTS)} / '
      f'{len(SELF_PASS2_PARTS)} ngày')
print('=' * 75)


In [ ]:
"""Cell 2: Các hàm trắc địa, tải NOAA Azure, lọc hàng hải & biến đổi."""


def to_naive_datetime(series: pd.Series) -> pd.Series:
    """Chuẩn hóa Series thời gian về UTC naive datetime."""
    dt = pd.to_datetime(series, errors='coerce')
    if getattr(dt.dt, 'tz', None) is not None:
        return dt.dt.tz_convert('UTC').dt.tz_localize(None)
    return dt


def distance_wgs84_np(
    lon1: np.ndarray,
    lat1: np.ndarray,
    lon2: np.ndarray,
    lat2: np.ndarray,
) -> np.ndarray:
    """Tính khoảng cách mặt phỏng cầu elip WGS84 (km), chuẩn kinh tuyến 180."""
    with np.errstate(invalid='ignore'):
        semi_major = 6378.1370
        flattening = 1.0 / 298.257223563
        semi_minor = semi_major * (1.0 - flattening)

        lat_mid = np.radians((lat1 + lat2) / 2.0)
        num = (
            (semi_major ** 2 * np.cos(lat_mid)) ** 2
            + (semi_minor ** 2 * np.sin(lat_mid)) ** 2
        )
        den = (
            (semi_major * np.cos(lat_mid)) ** 2
            + (semi_minor * np.sin(lat_mid)) ** 2
        )
        r_eff = np.sqrt(num / den)

        lon1_r, lat1_r, lon2_r, lat2_r = map(
            np.radians, [lon1, lat1, lon2, lat2]
        )
        dlat = lat2_r - lat1_r
        dlon = (lon2_r - lon1_r + np.pi) % (2.0 * np.pi) - np.pi
        hav = (
            np.sin(dlat / 2.0) ** 2
            + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2.0) ** 2
        )
        angle = 2.0 * np.arcsin(np.sqrt(np.clip(hav, 0.0, 1.0)))
        return (r_eff * angle).astype(np.float32)


def download_noaa_day_robust(
    day_str: str,
    max_retries: int = 3,
) -> Optional[str]:
    """Tải 1 tệp .csv.zst từ NOAA Azure Blob theo cơ chế nguyên tử."""
    fname = f'ais-{day_str}.csv.zst'
    fpath = os.path.join(RAW_TEMP_DIR, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) > 25 * 1024 * 1024:
        return fpath

    url = BASE_URL + fname
    tmp_path = fpath + '.downloading'
    for attempt in range(1, max_retries + 1):
        try:
            req = urllib.request.Request(
                url, headers={'User-Agent': 'Mozilla/5.0'}
            )
            with urllib.request.urlopen(req, timeout=120) as resp, open(
                tmp_path, 'wb'
            ) as out_f:
                while True:
                    chunk = resp.read(DOWNLOAD_BUFFER_SIZE)
                    if not chunk:
                        break
                    out_f.write(chunk)
            if (
                os.path.exists(tmp_path)
                and os.path.getsize(tmp_path) > 10 * 1024 * 1024
            ):
                os.replace(tmp_path, fpath)
                return fpath
        except Exception as exc:
            if os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except OSError:
                    pass
            if attempt == max_retries:
                print(f'  [Lỗi tải ngày {day_str}]: {exc}')
                return None
            time.sleep(2 * attempt)
    return None


def compute_weighted_median(
    values: np.ndarray,
    weights: np.ndarray,
) -> float:
    """Tính trung vị có trọng số chính xác từ mảng giá trị và tần suất."""
    if len(values) == 0:
        return float('nan')

    order = np.argsort(values, kind='mergesort')
    v_sorted = values[order]
    w_sorted = weights[order]
    cum_w = np.cumsum(w_sorted)
    total_w = int(cum_w[-1])
    if total_w <= 0:
        return float('nan')

    low_idx = int(np.searchsorted(cum_w, (total_w - 1) // 2, side='right'))
    high_idx = int(np.searchsorted(cum_w, total_w // 2, side='right'))
    return float(0.5 * (v_sorted[low_idx] + v_sorted[high_idx]))


class SlidingBatchDeduplicator:
    """Khử trùng lặp khóa (mmsi, base_date_time) xuyên biên batch."""

    def __init__(
        self,
        keep_days: int = 7,
        max_cached_keys: int = 3_000_000,
    ) -> None:
        """Khởi tạo bộ theo dõi trùng lặp với giới hạn bộ nhớ an toàn."""
        self.keep_days = keep_days
        self.max_cached_keys = max_cached_keys
        self.seen_by_day: Dict[int, Set[Tuple[int, int]]] = {}
        self.max_cutoff_day = -1
        self.total_keys = 0

    def filter_unique(self, df: pd.DataFrame) -> pd.DataFrame:
        """Loại bỏ các dòng trùng cặp (mmsi, base_date_time)."""
        if len(df) == 0:
            return df

        df_dedup = df.drop_duplicates(
            subset=['mmsi', 'base_date_time'],
            keep='first',
        )
        mmsi_arr = df_dedup['mmsi'].to_numpy(dtype=np.int64)
        ts_us = (
            to_naive_datetime(df_dedup['base_date_time'])
            .to_numpy(dtype='datetime64[us]')
            .view(np.int64)
        )
        day_arr = (ts_us // 86_400_000_000).astype(np.int64)

        cutoff_day = int(day_arr.min()) - self.keep_days
        if cutoff_day > self.max_cutoff_day:
            self.max_cutoff_day = cutoff_day
            old_days = [
                d for d in self.seen_by_day if d < self.max_cutoff_day
            ]
            for old_d in old_days:
                self.total_keys -= len(self.seen_by_day[old_d])
                del self.seen_by_day[old_d]

        keep_mask = np.ones(len(df_dedup), dtype=bool)
        seen_dict = self.seen_by_day
        for idx, (day_i, m_i, t_i) in enumerate(
            zip(day_arr.tolist(), mmsi_arr.tolist(), ts_us.tolist())
        ):
            key = (m_i, t_i)
            day_set = seen_dict.get(day_i)
            if day_set is None:
                day_set = set()
                seen_dict[day_i] = day_set
                day_set.add(key)
                self.total_keys += 1
            elif key in day_set:
                keep_mask[idx] = False
            else:
                day_set.add(key)
                self.total_keys += 1

        if self.total_keys > self.max_cached_keys:
            for old_d in sorted(list(seen_dict.keys())):
                if self.total_keys <= self.max_cached_keys:
                    break
                self.total_keys -= len(seen_dict[old_d])
                del seen_dict[old_d]

        return df_dedup.iloc[keep_mask].copy()


def apply_maritime_base_filters(
    df_raw: pd.DataFrame,
    atd_prev_map: Dict[str, object],
    is_train: bool,
) -> pd.DataFrame:
    """Lọc hàng hải đầu vào và kiểm soát nhãn riêng cho Q1 vs Q2..Q4.

    Trên Q1 (is_train=True): lọc thêm Target_Hours > 0 và
    Dist_To_Berth_NM / Target_Hours >= 1.62 knots.
    Trên Q2, Q3, Q4 (is_train=False): tuyệt đối KHÔNG lọc theo
    Target_Hours để giữ nguyên phân phối kiểm định thực tế.
    """
    if len(df_raw) == 0:
        return pd.DataFrame()

    bdt_naive = to_naive_datetime(df_raw['base_date_time'])
    sog_val = pd.to_numeric(df_raw['sog'], errors='coerce')
    st_val = pd.to_numeric(df_raw['status'], errors='coerce')
    vt_val = (
        pd.to_numeric(df_raw['vessel_type'], errors='coerce')
        .fillna(-1)
        .astype(np.int32)
    )

    atd_prev_series = to_naive_datetime(df_raw['trip_id'].map(atd_prev_map))
    valid_after_prev = atd_prev_series.isna() | (bdt_naive > atd_prev_series)

    base_mask = (
        sog_val.between(1.0, 40.0)
        & (~st_val.isin([1.0, 5.0]))
        & df_raw['latitude'].between(15.0, 75.0)
        & df_raw['longitude'].between(-180.0, -65.0)
        & vt_val.between(70, 89)
        & df_raw['berth_lon'].notna()
        & df_raw['berth_lat'].notna()
        & bdt_naive.notna()
        & valid_after_prev
    )

    clean = df_raw.loc[base_mask].copy()
    if len(clean) == 0:
        return clean

    clean['base_date_time'] = bdt_naive.loc[clean.index]
    clean['ata_time'] = to_naive_datetime(clean['ata_time'])
    clean['vessel_type'] = vt_val.loc[clean.index].to_numpy(dtype=np.int32)
    clean['Is_Tanker'] = (clean['vessel_type'] >= 80).astype(np.int32)

    dist_km = distance_wgs84_np(
        clean['longitude'].to_numpy(dtype=np.float32),
        clean['latitude'].to_numpy(dtype=np.float32),
        clean['berth_lon'].to_numpy(dtype=np.float32),
        clean['berth_lat'].to_numpy(dtype=np.float32),
    )
    clean['Dist_To_Berth_NM'] = (
        dist_km / np.float32(1.852)
    ).astype(np.float32)

    max_dist = np.where(
        clean['Is_Tanker'].to_numpy() == 1,
        MAX_DIST_TANKER_NM,
        MAX_DIST_CARGO_NM,
    )
    dist_mask = clean['Dist_To_Berth_NM'].to_numpy() <= max_dist
    clean = clean.iloc[dist_mask].copy()
    if len(clean) == 0:
        return clean

    if is_train:
        target_h = pd.to_numeric(clean['Target_Hours'], errors='coerce')
        implied_spd = np.where(
            target_h > 0.0,
            clean['Dist_To_Berth_NM'].to_numpy() / target_h.to_numpy(),
            -1.0,
        )
        train_lbl_mask = (
            (target_h > 0.0) & (implied_spd >= MIN_IMPLIED_SPEED_KNOTS)
        )
        clean = clean.loc[train_lbl_mask].copy()

    return clean


def apply_q1_fitted_transform(
    df_clean: pd.DataFrame,
    artifacts: dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Điền khuyết phân tầng và tính đặc trưng dùng tham số fit trên Q1.

    Dùng trực tiếp cho Q1 khi xuất dữ liệu, hoặc gọi trong thực nghiệm
    để biến đổi động Q2, Q3, Q4 bằng tham số Baseline của Q1.
    """
    if len(df_clean) == 0:
        return pd.DataFrame(columns=COLS_BASE), pd.DataFrame(
            columns=COLS_PREP
        )

    df = df_clean.copy()
    n_rows = len(df)
    mmsi_s = df['mmsi']
    vt_s = df['vessel_type']
    is_tanker = df['Is_Tanker'].to_numpy() == 1

    mmsi_med_root = artifacts.get('mmsi_medians', {})
    type_med_root = artifacts.get('type_medians', {})
    grp_med_root = artifacts.get('group_medians', {})

    imp_flags = []
    for col in ['length', 'width', 'draft']:
        vals_arr = pd.to_numeric(df[col], errors='coerce').to_numpy(
            dtype=np.float32, copy=True
        )
        if col == 'draft':
            bad_arr = (
                np.isnan(vals_arr) | (vals_arr <= 0.0) | (vals_arr > 25.5)
            )
        else:
            bad_arr = np.isnan(vals_arr) | (vals_arr <= 0.0)

        flag_col = np.zeros(n_rows, dtype=np.int32)
        if bad_arr.any():
            mmsi_map = mmsi_med_root.get(col, {})
            mmsi_fill = pd.to_numeric(
                mmsi_s.map(mmsi_map), errors='coerce'
            ).to_numpy(dtype=np.float64)
            take_mmsi = bad_arr & (~np.isnan(mmsi_fill))
            if take_mmsi.any():
                vals_arr[take_mmsi] = mmsi_fill[take_mmsi].astype(np.float32)
                flag_col[take_mmsi] = 1

            still_bad = bad_arr & (~take_mmsi)
            if still_bad.any():
                raw_vt_map = type_med_root.get(col, {})
                vt_map = {int(k): v for k, v in raw_vt_map.items()}
                type_fill = pd.to_numeric(
                    vt_s.map(vt_map), errors='coerce'
                ).to_numpy(dtype=np.float64)
                take_type = still_bad & (~np.isnan(type_fill))
                if take_type.any():
                    vals_arr[take_type] = type_fill[take_type].astype(
                        np.float32
                    )
                    flag_col[take_type] = 2

                remain = still_bad & (~take_type)
                if remain.any():
                    t_def = grp_med_root.get('tanker', {}).get(col, np.nan)
                    c_def = grp_med_root.get('cargo', {}).get(col, np.nan)
                    grp_fill = np.where(is_tanker, t_def, c_def).astype(
                        np.float32
                    )
                    vals_arr[remain] = grp_fill[remain]
                    flag_col[remain] = 3

        df[col] = vals_arr
        imp_flags.append(flag_col)

    imputation_flag = np.maximum.reduce(imp_flags).astype(np.int32)

    cog_vals = (
        pd.to_numeric(df['cog'], errors='coerce')
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    hdg_vals = (
        pd.to_numeric(df['heading'], errors='coerce')
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    df['cog'] = cog_vals
    df['heading'] = hdg_vals

    st_vals = pd.to_numeric(df['status'], errors='coerce')
    df['status'] = st_vals.fillna(
        np.float32(artifacts['status_mode'])
    ).astype(np.float32)

    df['cargo'] = pd.to_numeric(df['cargo'], errors='coerce').astype(
        np.float32
    )
    df['Area_Proxy'] = (df['length'] * df['width']).astype(np.float32)
    df['Volume_Proxy'] = (df['Area_Proxy'] * df['draft']).astype(np.float32)

    df['mmsi'] = df['mmsi'].astype(np.int64)
    df['trip_id'] = df['trip_id'].astype(str)
    df['base_date_time'] = to_naive_datetime(df['base_date_time'])
    df['ata_time'] = to_naive_datetime(df['ata_time'])
    df['longitude'] = df['longitude'].astype(np.float32)
    df['latitude'] = df['latitude'].astype(np.float32)
    df['sog'] = df['sog'].astype(np.float32)
    df['berth_lon'] = df['berth_lon'].astype(np.float32)
    df['berth_lat'] = df['berth_lat'].astype(np.float32)
    df['Target_Hours'] = df['Target_Hours'].astype(np.float32)

    df_base_22 = df[COLS_BASE].copy()
    df_prep_27 = df_base_22.copy()

    cog_rad = np.radians(cog_vals)
    hdg_rad = np.radians(hdg_vals)
    df_prep_27['cog_sin'] = np.sin(cog_rad).astype(np.float32)
    df_prep_27['cog_cos'] = np.cos(cog_rad).astype(np.float32)
    df_prep_27['heading_sin'] = np.sin(hdg_rad).astype(np.float32)
    df_prep_27['heading_cos'] = np.cos(hdg_rad).astype(np.float32)
    df_prep_27['imputation_flag'] = imputation_flag
    df_prep_27 = df_prep_27[COLS_PREP].copy()

    return df_base_22, df_prep_27


def build_unimputed_features_for_eval_quarter(
    df_clean: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Tạo đặc trưng hình học/lượng giác cho Q2..Q4 nhưng GIỮ NGUYÊN NaN.

    Tuyệt đối KHÔNG điền khuyết các cột thân vỏ, status, cog, heading.
    Các giá trị phi vật lý (length/width <= 0, draft <= 0 hoặc > 25.5)
    được chuẩn hóa về NaN. Cờ imputation_flag = -1 đánh dấu các dòng
    cần được điền khuyết khi chạy thực nghiệm; 0 là dòng nguyên vẹn.
    """
    if len(df_clean) == 0:
        return pd.DataFrame(columns=COLS_BASE), pd.DataFrame(
            columns=COLS_PREP
        )

    df = df_clean.copy()

    l_arr = pd.to_numeric(df['length'], errors='coerce').to_numpy(
        dtype=np.float32, copy=True
    )
    w_arr = pd.to_numeric(df['width'], errors='coerce').to_numpy(
        dtype=np.float32, copy=True
    )
    d_arr = pd.to_numeric(df['draft'], errors='coerce').to_numpy(
        dtype=np.float32, copy=True
    )

    bad_l = np.isnan(l_arr) | (l_arr <= 0.0)
    bad_w = np.isnan(w_arr) | (w_arr <= 0.0)
    bad_d = np.isnan(d_arr) | (d_arr <= 0.0) | (d_arr > 25.5)

    l_arr[bad_l] = np.nan
    w_arr[bad_w] = np.nan
    d_arr[bad_d] = np.nan

    df['length'] = l_arr
    df['width'] = w_arr
    df['draft'] = d_arr
    has_missing_hull = bad_l | bad_w | bad_d

    # Giữ nguyên NaN gốc của cog, heading, status, cargo
    cog_vals = pd.to_numeric(df['cog'], errors='coerce').to_numpy(
        dtype=np.float32
    )
    hdg_vals = pd.to_numeric(df['heading'], errors='coerce').to_numpy(
        dtype=np.float32
    )
    df['cog'] = cog_vals
    df['heading'] = hdg_vals
    df['status'] = pd.to_numeric(df['status'], errors='coerce').astype(
        np.float32
    )
    df['cargo'] = pd.to_numeric(df['cargo'], errors='coerce').astype(
        np.float32
    )

    df['Area_Proxy'] = (df['length'] * df['width']).astype(np.float32)
    df['Volume_Proxy'] = (df['Area_Proxy'] * df['draft']).astype(np.float32)

    df['mmsi'] = df['mmsi'].astype(np.int64)
    df['trip_id'] = df['trip_id'].astype(str)
    df['base_date_time'] = to_naive_datetime(df['base_date_time'])
    df['ata_time'] = to_naive_datetime(df['ata_time'])
    df['longitude'] = df['longitude'].astype(np.float32)
    df['latitude'] = df['latitude'].astype(np.float32)
    df['sog'] = df['sog'].astype(np.float32)
    df['berth_lon'] = df['berth_lon'].astype(np.float32)
    df['berth_lat'] = df['berth_lat'].astype(np.float32)
    df['Target_Hours'] = df['Target_Hours'].astype(np.float32)

    df_base_22 = df[COLS_BASE].copy()
    df_prep_27 = df_base_22.copy()

    with np.errstate(invalid='ignore'):
        cog_rad = np.radians(cog_vals)
        hdg_rad = np.radians(hdg_vals)
        df_prep_27['cog_sin'] = np.sin(cog_rad).astype(np.float32)
        df_prep_27['cog_cos'] = np.cos(cog_rad).astype(np.float32)
        df_prep_27['heading_sin'] = np.sin(hdg_rad).astype(np.float32)
        df_prep_27['heading_cos'] = np.cos(hdg_rad).astype(np.float32)

    df_prep_27['imputation_flag'] = np.where(
        has_missing_hull, -1, 0
    ).astype(np.int32)
    df_prep_27 = df_prep_27[COLS_PREP].copy()

    return df_base_22, df_prep_27


### BƯỚC 0 (TẢI & GHÉP NHÃN GỐC): PASS 1 + PASS 2 TỪ NOAA AZURE BLOB (TỰ KẾ THỪA CHECKPOINT)

- Nếu **đã có sẵn** `noaa_ais_2025_raw_labeled.parquet` hoặc cặp `01_ban_tho_18_thuoc_tinh/{train,test}.parquet` trong `/kaggle/input` hoặc `/kaggle/working` (và `FORCE_REPROCESS_ALL == False`), hệ thống sẽ tự động kế thừa ngay lập tức để **tiết kiệm 6-8 giờ** tải 365 tệp `.csv.zst`.
- Nếu chạy **từ đầu (Cold-Start)** hoặc bật `FORCE_REPROCESS_ALL = True`:
  + **Pass 1:** Quét sự kiện cập bến `status == 5.0` & `sog < 0.5 kn`, phân cụm bến chống băm nhỏ (`Anti-Overfragmentation`) và lưu checkpoint `ata_catalog.parquet`.
  + **Pass 2:** Ghép `pd.merge_asof` với `ata_catalog.parquet` theo từng ngày (`part_raw_*.parquet`) và hợp nhất thành tệp thô gán nhãn 18 cột `noaa_ais_2025_raw_labeled.parquet`.


In [ ]:
"""Cell 4: Thực thi Pass 1 + Pass 2 (nếu cần) & Cung cấp nguồn Raw 18 cột."""


def run_pass1_and_pass2_from_noaa_blob() -> Tuple[str, Optional[str]]:
    """Tải dữ liệu NOAA 2025 từ Azure Blob, chạy Pass 1 và Pass 2."""
    print('=' * 75)
    print('PASS 1: QUÉT SỰ KIỆN CẬP BẾN 365 NGÀY (SELF-CHECKPOINTING)')
    print('=' * 75)

    ata_catalog = None
    if (
        (RUN_MODE == 'PASS2_ONLY' or not FORCE_REPROCESS_ALL)
        and SELF_CATALOG_FILE is not None
        and os.path.exists(SELF_CATALOG_FILE)
    ):
        print(f'[✓] Nạp ata_catalog có sẵn: {SELF_CATALOG_FILE}')
        ata_catalog = pd.read_parquet(SELF_CATALOG_FILE)
    else:
        if RUN_MODE == 'PASS2_ONLY':
            raise RuntimeError(
                'PASS2_ONLY yêu cầu có sẵn ata_catalog.parquet!'
            )
        t_p1 = time.time()
        moored_days: List[pd.DataFrame] = []
        avail_p1 = get_available_pass1_checkpoints()
        print(
            f'  • Pass 1 checkpoints sẵn có: {len(avail_p1)}/{len(date_list)}'
        )

        wanted_p1 = {
            'mmsi',
            'base_date_time',
            'basedatetime',
            'longitude',
            'lon',
            'latitude',
            'lat',
            'sog',
            'status',
            'vessel_type',
            'vesseltype',
        }

        for idx, day_str in enumerate(date_list):
            ckpt_f = os.path.join(
                PASS1_CHECKPOINT_DIR, f'moored_{day_str}.parquet'
            )
            if not FORCE_REPROCESS_ALL and day_str in avail_p1:
                try:
                    df_d = pd.read_parquet(avail_p1[day_str])
                    if len(df_d) > 0:
                        moored_days.append(df_d)
                    continue
                except Exception:
                    pass

            fpath = download_noaa_day_robust(day_str)
            if fpath is None:
                continue

            day_events = []
            try:
                with pa.input_stream(fpath, compression='zstd') as stream:
                    for chunk in pd.read_csv(
                        stream,
                        usecols=lambda c: c.lower() in wanted_p1,
                        chunksize=CHUNK_SIZE,
                    ):
                        cmap = {c.lower(): c for c in chunk.columns}
                        m_col = cmap.get('mmsi')
                        t_col = cmap.get('base_date_time') or cmap.get(
                            'basedatetime'
                        )
                        lat_col = cmap.get('latitude') or cmap.get('lat')
                        lon_col = cmap.get('longitude') or cmap.get('lon')
                        sog_col = cmap.get('sog')
                        st_col = cmap.get('status')
                        vt_col = cmap.get('vessel_type') or cmap.get(
                            'vesseltype'
                        )

                        vt_num = pd.to_numeric(chunk[vt_col], errors='coerce')
                        st_num = pd.to_numeric(chunk[st_col], errors='coerce')
                        sog_num = pd.to_numeric(
                            chunk[sog_col], errors='coerce'
                        )
                        lat_num = pd.to_numeric(
                            chunk[lat_col], errors='coerce'
                        )
                        lon_num = pd.to_numeric(
                            chunk[lon_col], errors='coerce'
                        )

                        mask = (
                            vt_num.between(70, 89)
                            & lat_num.between(15.0, 75.0)
                            & lon_num.between(-180.0, -65.0)
                            & (st_num == 5.0)
                            & (sog_num < 0.5)
                        )
                        if mask.any():
                            sub = pd.DataFrame({
                                'mmsi': pd.to_numeric(
                                    chunk.loc[mask, m_col], errors='coerce'
                                ).astype(np.int64),
                                'base_date_time': pd.to_datetime(
                                    chunk.loc[mask, t_col],
                                    format='ISO8601',
                                    errors='coerce',
                                ),
                                'longitude': lon_num.loc[mask].astype(
                                    np.float32
                                ),
                                'latitude': lat_num.loc[mask].astype(
                                    np.float32
                                ),
                            })
                            sub = sub[sub['base_date_time'].notna()].copy()
                            sub['dt_r'] = sub['base_date_time'].dt.floor(
                                '30min'
                            )
                            sub = sub.drop_duplicates(
                                subset=['mmsi', 'dt_r']
                            ).drop(columns=['dt_r'])
                            day_events.append(sub)

                tmp_f = ckpt_f + '.tmp'
                if day_events:
                    df_d = pd.concat(day_events, ignore_index=True)
                    df_d.to_parquet(tmp_f, index=False)
                    os.replace(tmp_f, ckpt_f)
                    moored_days.append(df_d)
                else:
                    pd.DataFrame(
                        columns=[
                            'mmsi',
                            'base_date_time',
                            'longitude',
                            'latitude',
                        ]
                    ).to_parquet(tmp_f, index=False)
                    os.replace(tmp_f, ckpt_f)
            except Exception as exc:
                print(f'  [Lỗi Pass 1 {day_str}]: {exc}')
            finally:
                if len(date_list) > 5 and os.path.exists(fpath):
                    os.remove(fpath)

            if (idx + 1) % 30 == 0 or (idx + 1) == len(date_list):
                el_min = (time.time() - t_p1) / 60.0
                print(
                    f'  [Pass 1] {idx + 1:3d}/{len(date_list)} ngày '
                    f'({el_min:5.1f} phút)'
                )

        if not moored_days:
            raise RuntimeError('Không phát hiện sự kiện buộc bến nào!')

        df_moored = (
            pd.concat(moored_days, ignore_index=True)
            .sort_values(['mmsi', 'base_date_time'])
            .reset_index(drop=True)
        )
        tgap = (
            df_moored['base_date_time']
            - df_moored.groupby('mmsi')['base_date_time'].shift(1)
        ).dt.total_seconds() / 3600.0
        plon = df_moored.groupby('mmsi')['longitude'].shift(1)
        plat = df_moored.groupby('mmsi')['latitude'].shift(1)
        dgap = distance_wgs84_np(
            df_moored['longitude'].to_numpy(dtype=np.float32),
            df_moored['latitude'].to_numpy(dtype=np.float32),
            plon.to_numpy(dtype=np.float32),
            plat.to_numpy(dtype=np.float32),
        )
        is_new = (
            tgap.isna()
            | (dgap >= 2.0)
            | (tgap >= 24.0)
            | ((tgap >= 8.0) & (dgap >= 0.5))
        )
        df_moored['call_id'] = is_new.cumsum()

        ata_catalog = (
            df_moored.groupby(['mmsi', 'call_id'])
            .agg(
                ata_time=('base_date_time', 'min'),
                atd_time=('base_date_time', 'max'),
                berth_lon=('longitude', 'median'),
                berth_lat=('latitude', 'median'),
                moored_points=('base_date_time', 'count'),
            )
            .reset_index()
        )
        ata_catalog = ata_catalog[ata_catalog['moored_points'] >= 2].copy()
        ata_catalog['trip_id'] = (
            ata_catalog['mmsi'].astype(str)
            + '_'
            + ata_catalog['ata_time'].dt.strftime('%Y%m%d_%H%M')
        )
        ata_catalog = ata_catalog.sort_values(
            ['mmsi', 'ata_time']
        ).reset_index(drop=True)
        ata_catalog['atd_prev'] = ata_catalog.groupby('mmsi')[
            'atd_time'
        ].shift(1)
        ata_catalog.to_parquet(CATALOG_PARQUET, index=False)
        del df_moored, moored_days
        gc.collect()

    print(f'[✓] Hoàn tất Pass 1: {len(ata_catalog):,} chuyến cập bến.')
    if RUN_MODE == 'PASS1_ONLY':
        return 'PASS1_ONLY', CATALOG_PARQUET

    print('=' * 75)
    print('PASS 2: TRÍCH XUẤT BẢN THÔ GẮN NHÃN 18 CỘT THEO TỪNG NGÀY')
    print('=' * 75)

    ata_lookup = ata_catalog[
        ['mmsi', 'trip_id', 'ata_time', 'berth_lon', 'berth_lat']
    ].sort_values('ata_time')
    cand_mmsi = set(ata_catalog['mmsi'].unique().tolist())

    avail_p2 = {} if FORCE_REPROCESS_ALL else get_available_pass2_parts()
    print(f'  • Pass 2 part_raw sẵn có: {len(avail_p2)}/{len(date_list)}')

    wanted_p2 = {
        'mmsi',
        'base_date_time',
        'basedatetime',
        'longitude',
        'lon',
        'latitude',
        'lat',
        'sog',
        'cog',
        'heading',
        'vessel_type',
        'vesseltype',
        'status',
        'length',
        'width',
        'draft',
        'cargo',
    }

    for idx, day_str in enumerate(date_list):
        p_raw_f = os.path.join(PASS2_PARTS_DIR, f'part_raw_{day_str}.parquet')
        if not FORCE_REPROCESS_ALL and day_str in avail_p2:
            continue

        fpath = download_noaa_day_robust(day_str)
        if fpath is None:
            continue

        raw_chunks = []
        try:
            with pa.input_stream(fpath, compression='zstd') as stream:
                for chunk in pd.read_csv(
                    stream,
                    usecols=lambda c: c.lower() in wanted_p2,
                    chunksize=CHUNK_SIZE,
                ):
                    cmap = {c.lower(): c for c in chunk.columns}
                    m_c = cmap.get('mmsi')
                    t_c = cmap.get('base_date_time') or cmap.get(
                        'basedatetime'
                    )
                    lat_c = cmap.get('latitude') or cmap.get('lat')
                    lon_c = cmap.get('longitude') or cmap.get('lon')
                    sog_c = cmap.get('sog')
                    cog_c = cmap.get('cog')
                    hdg_c = cmap.get('heading')
                    vt_c = cmap.get('vessel_type') or cmap.get('vesseltype')
                    st_c = cmap.get('status')
                    l_c = cmap.get('length')
                    w_c = cmap.get('width')
                    d_c = cmap.get('draft')
                    cg_c = cmap.get('cargo')

                    mmsi_num = pd.to_numeric(chunk[m_c], errors='coerce')
                    df_sub = chunk[mmsi_num.isin(cand_mmsi)].copy()
                    if len(df_sub) == 0:
                        continue

                    df_sub['mmsi'] = (
                        pd.to_numeric(df_sub[m_c], errors='coerce')
                        .fillna(0)
                        .astype(np.int64)
                    )
                    df_sub['base_date_time'] = pd.to_datetime(
                        df_sub[t_c], format='ISO8601', errors='coerce'
                    )
                    df_sub = df_sub[df_sub['base_date_time'].notna()]
                    df_sub = df_sub.sort_values('base_date_time')
                    if len(df_sub) == 0:
                        continue

                    merged = pd.merge_asof(
                        df_sub,
                        ata_lookup,
                        by='mmsi',
                        left_on='base_date_time',
                        right_on='ata_time',
                        direction='forward',
                    )
                    tgt_h = (
                        merged['ata_time'] - merged['base_date_time']
                    ).dt.total_seconds() / 3600.0

                    df_part = pd.DataFrame({
                        'mmsi': merged['mmsi'].astype(np.int64),
                        'trip_id': merged['trip_id']
                        .fillna('UNKNOWN')
                        .astype(str),
                        'base_date_time': to_naive_datetime(
                            merged['base_date_time']
                        ),
                        'longitude': pd.to_numeric(
                            merged[lon_c], errors='coerce'
                        ).astype(np.float32),
                        'latitude': pd.to_numeric(
                            merged[lat_c], errors='coerce'
                        ).astype(np.float32),
                        'sog': pd.to_numeric(
                            merged[sog_c], errors='coerce'
                        ).astype(np.float32),
                        'cog': pd.to_numeric(
                            merged[cog_c], errors='coerce'
                        ).astype(np.float32),
                        'heading': pd.to_numeric(
                            merged[hdg_c], errors='coerce'
                        ).astype(np.float32),
                        'status': pd.to_numeric(
                            merged[st_c], errors='coerce'
                        ).astype(np.float32),
                        'vessel_type': pd.to_numeric(
                            merged[vt_c], errors='coerce'
                        ).astype(np.float32),
                        'length': pd.to_numeric(
                            merged[l_c], errors='coerce'
                        ).astype(np.float32),
                        'width': pd.to_numeric(
                            merged[w_c], errors='coerce'
                        ).astype(np.float32),
                        'draft': pd.to_numeric(
                            merged[d_c], errors='coerce'
                        ).astype(np.float32),
                        'cargo': pd.to_numeric(
                            merged[cg_c], errors='coerce'
                        ).astype(np.float32),
                        'berth_lon': merged['berth_lon'].astype(np.float32),
                        'berth_lat': merged['berth_lat'].astype(np.float32),
                        'ata_time': to_naive_datetime(merged['ata_time']),
                        'Target_Hours': tgt_h.astype(np.float32),
                    })
                    raw_chunks.append(df_part[COLS_RAW])

            t_df = (
                pd.concat(raw_chunks, ignore_index=True)
                if raw_chunks
                else pd.DataFrame(columns=COLS_RAW)
            )
            tbl = pa.Table.from_pandas(
                t_df, schema=SCHEMA_RAW_LABELED, preserve_index=False
            )
            pq.write_table(tbl, p_raw_f + '.tmp', compression='snappy')
            os.replace(p_raw_f + '.tmp', p_raw_f)
            avail_p2[day_str] = p_raw_f
        except Exception as exc:
            print(f'  [Lỗi Pass 2 {day_str}]: {exc}')
        finally:
            if len(date_list) > 5 and os.path.exists(fpath):
                os.remove(fpath)

        if (idx + 1) % 30 == 0 or (idx + 1) == len(date_list):
            print(
                f'  [Pass 2] Đã xong: {len(avail_p2):3d}/{len(date_list)} ngày'
            )

    print('[*] Hợp nhất các part_raw_*.parquet -> raw_labeled.parquet...')
    if os.path.exists(RAW_LABELED_WORK_PARQUET):
        os.remove(RAW_LABELED_WORK_PARQUET)

    all_p2 = get_available_pass2_parts()
    writer_raw = pq.ParquetWriter(
        RAW_LABELED_WORK_PARQUET,
        SCHEMA_RAW_LABELED,
        compression=PARQUET_COMPRESSION,
    )
    try:
        for d_key in sorted(all_p2.keys()):
            tbl_p = pq.read_table(all_p2[d_key])
            if tbl_p.num_rows > 0:
                writer_raw.write_table(tbl_p)
            if CLEANUP_TEMP_AFTER_MERGE and all_p2[d_key].startswith(
                BASE_WORK_DIR
            ):
                os.remove(all_p2[d_key])
    finally:
        writer_raw.close()

    return 'SINGLE_RAW', RAW_LABELED_WORK_PARQUET


# KIỂM TRA CHECKPOINT HOẶC CHẠY PASS 1 + PASS 2 TỪ ĐẦU
if not FORCE_REPROCESS_ALL and RAW_SOURCE_MODE in (
    'SINGLE_RAW',
    'SPLIT_PAIR',
):
    print(f'[CHECKPOINT HIT] Tận dụng nguồn thô 18 cột sẵn có: '
          f'{RAW_SOURCE_MODE}')
    for src_p in RAW_SOURCE_PATHS:
        pf_chk = pq.ParquetFile(src_p)
        sz_gb = os.path.getsize(src_p) / (1024 ** 3)
        print(
            f'  • {os.path.basename(src_p)}: '
            f'{pf_chk.metadata.num_rows:,} dòng | '
            f'{pf_chk.metadata.num_columns} cột | {sz_gb:.2f} GiB'
        )
else:
    mode_res, res_path = run_pass1_and_pass2_from_noaa_blob()
    if mode_res == 'PASS1_ONLY':
        raise SystemExit('Dừng theo cấu hình RUN_MODE = PASS1_ONLY.')
    RAW_SOURCE_MODE = 'SINGLE_RAW'
    RAW_SOURCE_PATHS = [res_path]
    SELF_CATALOG_FILE = (
        CATALOG_PARQUET if os.path.isfile(CATALOG_PARQUET) else None
    )


def get_total_raw_18_rows() -> int:
    """Trả về tổng số dòng dữ liệu thô 18 cột từ nguồn hiện hành."""
    return sum(
        pq.ParquetFile(p).metadata.num_rows for p in RAW_SOURCE_PATHS
    )


def iter_raw_18_batches(
    columns: Optional[List[str]] = None,
    batch_size: int = BATCH_SIZE_SPLIT,
) -> Iterator[pd.DataFrame]:
    """Generator đọc streaming lần lượt từ nguồn thô 18 cột."""
    for src_p in RAW_SOURCE_PATHS:
        pf = pq.ParquetFile(src_p)
        for batch in pf.iter_batches(
            batch_size=batch_size, columns=columns
        ):
            yield batch.to_pandas()


### BƯỚC 1: KHÔI PHỤC `atd_prev` VÀ PHÂN CHIA 4 QUÝ ĐỘC LẬP TRƯỚC KHI TIỀN XỬ LÝ

Quá trình này thực hiện phân chia **trước khi fit hay transform bất kỳ thống kê nào**:
1. Xác định mốc rời cảng chuyến liền trước (`atd_prev`) của mỗi `trip_id` từ danh bạ buộc bến (`ata_catalog`).
2. Quét biên thời gian hành trình `[min_voyage_time, max_voyage_time]` (sau `atd_prev`).
3. Phân rã thành 4 tập hành trình thuần túy (`Q1_PURE`, `Q2_PURE`, `Q3_PURE`, `Q4_PURE`), loại bỏ triệt để các chuyến vượt biên giới quý (`CROSSING_Q1_Q2_EXCLUDED`, `CROSSING_Q2_Q3_EXCLUDED`, `CROSSING_Q3_Q4_EXCLUDED`) để khóa giao điểm tập chuyến giữa 4 quý bằng `0`.


In [ ]:
"""Cell 6: Bước 1 - Phân rã hành trình 4 quý & Khóa giao điểm bằng 0."""

t_step1 = time.time()
Q1_START = pd.Timestamp(QUARTER_SPECS['Q1']['start'])
Q2_START = pd.Timestamp(QUARTER_SPECS['Q2']['start'])
Q3_START = pd.Timestamp(QUARTER_SPECS['Q3']['start'])
Q4_START = pd.Timestamp(QUARTER_SPECS['Q4']['start'])
Q4_END = pd.Timestamp(QUARTER_SPECS['Q4']['end'])

print('=' * 75)
print('BƯỚC 1: KHÔI PHỤC atd_prev & PHÂN LOẠI HÀNH TRÌNH 4 QUÝ (PRE-SPLIT)')
print('=' * 75)

if (
    not FORCE_REPROCESS_ALL
    and SELF_SPLIT_MANIFEST is not None
    and SELF_CATALOG_RECON is not None
):
    print(f'[✓] Kế thừa Manifest gốc: {SELF_SPLIT_MANIFEST}')
    df_manifest = pd.read_parquet(SELF_SPLIT_MANIFEST)
    df_recon = pd.read_parquet(SELF_CATALOG_RECON)
    atd_prev_map = dict(
        zip(
            df_recon['trip_id'].astype(str),
            to_naive_datetime(df_recon['atd_prev']),
        )
    )
    if os.path.abspath(SELF_CATALOG_RECON) != os.path.abspath(
        CATALOG_RECON_PARQUET
    ):
        shutil.copyfile(SELF_CATALOG_RECON, CATALOG_RECON_PARQUET)
else:
    total_rows = get_total_raw_18_rows()
    has_cat = (
        SELF_CATALOG_FILE is not None
        and os.path.exists(SELF_CATALOG_FILE)
    )
    need_moored = not has_cat
    moored_parts: List[pd.DataFrame] = []
    raw_trips_parts: List[pd.DataFrame] = []
    scan_cols_1 = [
        'mmsi',
        'trip_id',
        'base_date_time',
        'longitude',
        'latitude',
        'sog',
        'status',
        'vessel_type',
        'ata_time',
    ]

    scanned = 0
    for b_idx, df_b in enumerate(
        iter_raw_18_batches(columns=scan_cols_1, batch_size=BATCH_SIZE_SCAN)
    ):
        scanned += len(df_b)
        df_b['base_date_time'] = to_naive_datetime(df_b['base_date_time'])
        df_b['ata_time'] = to_naive_datetime(df_b['ata_time'])

        valid_t = df_b['ata_time'].notna() & (df_b['trip_id'] != 'UNKNOWN')
        if valid_t.any():
            raw_trips_parts.append(
                df_b.loc[valid_t, ['trip_id', 'mmsi', 'ata_time']]
                .drop_duplicates(subset=['trip_id'])
            )

        if need_moored:
            vt_b = pd.to_numeric(df_b['vessel_type'], errors='coerce')
            sog_b = pd.to_numeric(df_b['sog'], errors='coerce')
            st_b = pd.to_numeric(df_b['status'], errors='coerce')
            m_mask = (
                vt_b.between(70, 89)
                & df_b['latitude'].between(15.0, 75.0)
                & df_b['longitude'].between(-180.0, -65.0)
                & (st_b == 5.0)
                & (sog_b < 0.5)
                & df_b['base_date_time'].notna()
            )
            if m_mask.any():
                m_sub = df_b.loc[
                    m_mask,
                    ['mmsi', 'base_date_time', 'longitude', 'latitude'],
                ].copy()
                m_sub['dt_r'] = m_sub['base_date_time'].dt.floor('30min')
                moored_parts.append(
                    m_sub.drop_duplicates(subset=['mmsi', 'dt_r'])
                )

        if (b_idx + 1) % 30 == 0 or scanned >= total_rows:
            print(f'  -> Quét danh bạ: {scanned:,}/{total_rows:,}')

    df_raw_trips = (
        pd.concat(raw_trips_parts, ignore_index=True)
        .drop_duplicates(subset=['trip_id'])
        .sort_values(['mmsi', 'ata_time'])
        .reset_index(drop=True)
    )
    del raw_trips_parts
    gc.collect()

    if has_cat:
        df_cat = pd.read_parquet(SELF_CATALOG_FILE)[
            ['trip_id', 'atd_time', 'atd_prev']
        ].copy()
        df_trips = df_raw_trips.merge(df_cat, on='trip_id', how='left')
        still_na = df_trips['atd_time'].isna()
        df_trips.loc[still_na, 'atd_time'] = df_trips.loc[
            still_na, 'ata_time'
        ]
        df_trips = df_trips.sort_values(['mmsi', 'ata_time']).reset_index(
            drop=True
        )
        miss_prev = df_trips['atd_prev'].isna()
        shifted_prev = df_trips.groupby('mmsi')['atd_time'].shift(1)
        df_trips.loc[miss_prev, 'atd_prev'] = shifted_prev.loc[miss_prev]
    else:
        df_moored = (
            pd.concat(moored_parts, ignore_index=True)
            .drop_duplicates(subset=['mmsi', 'dt_r'])
            .drop(columns=['dt_r'])
            .sort_values(['mmsi', 'base_date_time'])
            .reset_index(drop=True)
        )
        del moored_parts
        gc.collect()

        tgap = (
            df_moored['base_date_time']
            - df_moored.groupby('mmsi')['base_date_time'].shift(1)
        ).dt.total_seconds() / 3600.0
        plon = df_moored.groupby('mmsi')['longitude'].shift(1)
        plat = df_moored.groupby('mmsi')['latitude'].shift(1)
        dgap = distance_wgs84_np(
            df_moored['longitude'].to_numpy(dtype=np.float32),
            df_moored['latitude'].to_numpy(dtype=np.float32),
            plon.to_numpy(dtype=np.float32),
            plat.to_numpy(dtype=np.float32),
        )
        is_new = (
            tgap.isna()
            | (dgap >= 2.0)
            | (tgap >= 24.0)
            | ((tgap >= 8.0) & (dgap >= 0.5))
        )
        df_moored['call_id'] = is_new.cumsum()

        df_cat = (
            df_moored.groupby(['mmsi', 'call_id'])
            .agg(
                ata_recon=('base_date_time', 'min'),
                atd_recon=('base_date_time', 'max'),
                moored_points=('base_date_time', 'count'),
            )
            .reset_index()
        )
        df_cat = df_cat[df_cat['moored_points'] >= 2].copy()
        df_cat['recon_tid'] = (
            df_cat['mmsi'].astype(str)
            + '_'
            + df_cat['ata_recon'].dt.strftime('%Y%m%d_%H%M')
        )
        del df_moored
        gc.collect()

        exact_map = df_cat.set_index('recon_tid')['atd_recon']
        df_trips = df_raw_trips.copy()
        df_trips['atd_time'] = df_trips['trip_id'].map(exact_map)

        miss_m = df_trips['atd_time'].isna()
        if miss_m.any():
            df_miss = df_trips.loc[
                miss_m, ['trip_id', 'mmsi', 'ata_time']
            ].sort_values('ata_time')
            df_c_sorted = df_cat[
                ['mmsi', 'ata_recon', 'atd_recon']
            ].sort_values('ata_recon')
            near = pd.merge_asof(
                df_miss,
                df_c_sorted,
                by='mmsi',
                left_on='ata_time',
                right_on='ata_recon',
                direction='nearest',
                tolerance=pd.Timedelta('45min'),
            ).set_index('trip_id')['atd_recon']
            df_trips.loc[miss_m, 'atd_time'] = df_trips.loc[
                miss_m, 'trip_id'
            ].map(near)

        still_nat = df_trips['atd_time'].isna()
        df_trips.loc[still_nat, 'atd_time'] = df_trips.loc[
            still_nat, 'ata_time'
        ]
        df_trips = df_trips.sort_values(['mmsi', 'ata_time']).reset_index(
            drop=True
        )
        df_trips['atd_prev'] = df_trips.groupby('mmsi')['atd_time'].shift(1)

    df_trips.to_parquet(
        CATALOG_RECON_PARQUET, index=False, compression=PARQUET_COMPRESSION
    )
    atd_prev_map = dict(
        zip(df_trips['trip_id'], to_naive_datetime(df_trips['atd_prev']))
    )

    trip_bounds: Dict[str, list] = {}
    scanned = 0
    for b_idx, df_b in enumerate(
        iter_raw_18_batches(
            columns=['trip_id', 'mmsi', 'base_date_time', 'ata_time'],
            batch_size=BATCH_SIZE_SCAN,
        )
    ):
        scanned += len(df_b)
        bdt = to_naive_datetime(df_b['base_date_time'])
        atd_p = to_naive_datetime(df_b['trip_id'].map(atd_prev_map))
        ok = (
            df_b['ata_time'].notna()
            & (df_b['trip_id'] != 'UNKNOWN')
            & (atd_p.isna() | (bdt > atd_p))
        )
        if ok.any():
            sub = df_b.loc[ok, ['trip_id']].copy()
            sub['bdt'] = bdt.loc[ok]
            grp = (
                sub.groupby('trip_id')
                .agg(min_t=('bdt', 'min'), max_t=('bdt', 'max'),
                     cnt=('bdt', 'count'))
                .reset_index()
            )
            for r in grp.itertuples(index=False):
                cur = trip_bounds.get(r.trip_id)
                if cur is None:
                    trip_bounds[r.trip_id] = [r.min_t, r.max_t, int(r.cnt)]
                else:
                    if r.min_t < cur[0]:
                        cur[0] = r.min_t
                    if r.max_t > cur[1]:
                        cur[1] = r.max_t
                    cur[2] += int(r.cnt)
        if (b_idx + 1) % 30 == 0 or scanned >= total_rows:
            print(f'  -> Xác định biên chuyến: {scanned:,}/{total_rows:,}')

    df_manifest = df_trips[
        ['trip_id', 'mmsi', 'ata_time', 'atd_time', 'atd_prev']
    ].copy()
    df_manifest['min_voyage_time'] = pd.to_datetime(
        df_manifest['trip_id'].map({k: v[0] for k, v in trip_bounds.items()})
    )
    df_manifest['max_voyage_time'] = pd.to_datetime(
        df_manifest['trip_id'].map({k: v[1] for k, v in trip_bounds.items()})
    )
    df_manifest['num_raw_voyage_rows'] = (
        df_manifest['trip_id']
        .map({k: v[2] for k, v in trip_bounds.items()})
        .fillna(0)
        .astype(np.int64)
    )
    del df_trips, df_raw_trips, trip_bounds
    gc.collect()

# PHÂN NHÓM 4 QUÝ THUẦN TÚY & LOẠI BỎ TRIỆT ĐỂ CROSSING TRIPS
df_manifest['min_voyage_time'] = to_naive_datetime(
    df_manifest['min_voyage_time']
)
df_manifest['max_voyage_time'] = to_naive_datetime(
    df_manifest['max_voyage_time']
)
df_manifest['ata_time'] = to_naive_datetime(df_manifest['ata_time'])

mask_obs = df_manifest['num_raw_voyage_rows'] > 0
m_q1 = (
    mask_obs
    & (df_manifest['min_voyage_time'] >= Q1_START)
    & (df_manifest['max_voyage_time'] < Q2_START)
    & (df_manifest['ata_time'] >= Q1_START)
    & (df_manifest['ata_time'] < Q2_START)
)
m_q2 = (
    mask_obs
    & (df_manifest['min_voyage_time'] >= Q2_START)
    & (df_manifest['max_voyage_time'] < Q3_START)
    & (df_manifest['ata_time'] >= Q2_START)
    & (df_manifest['ata_time'] < Q3_START)
)
m_q3 = (
    mask_obs
    & (df_manifest['min_voyage_time'] >= Q3_START)
    & (df_manifest['max_voyage_time'] < Q4_START)
    & (df_manifest['ata_time'] >= Q3_START)
    & (df_manifest['ata_time'] < Q4_START)
)
m_q4 = (
    mask_obs
    & (df_manifest['min_voyage_time'] >= Q4_START)
    & (df_manifest['max_voyage_time'] < Q4_END)
    & (df_manifest['ata_time'] >= Q4_START)
    & (df_manifest['ata_time'] < Q4_END)
)

df_manifest['quarter_category'] = 'OTHER_DROPPED'
df_manifest.loc[m_q1, 'quarter_category'] = 'Q1_PURE'
df_manifest.loc[m_q2, 'quarter_category'] = 'Q2_PURE'
df_manifest.loc[m_q3, 'quarter_category'] = 'Q3_PURE'
df_manifest.loc[m_q4, 'quarter_category'] = 'Q4_PURE'
df_manifest.loc[~mask_obs, 'quarter_category'] = 'DROPPED_NO_VOYAGE_OBS'

m_cx_12 = (
    (df_manifest['quarter_category'] == 'OTHER_DROPPED')
    & (df_manifest['min_voyage_time'] < Q2_START)
    & (
        (df_manifest['max_voyage_time'] >= Q2_START)
        | (df_manifest['ata_time'] >= Q2_START)
    )
    & (df_manifest['max_voyage_time'] < Q3_START)
    & (df_manifest['ata_time'] < Q3_START)
)
df_manifest.loc[m_cx_12, 'quarter_category'] = 'CROSSING_Q1_Q2_EXCLUDED'

m_cx_23 = (
    (df_manifest['quarter_category'] == 'OTHER_DROPPED')
    & (df_manifest['min_voyage_time'] >= Q2_START)
    & (df_manifest['min_voyage_time'] < Q3_START)
    & (
        (df_manifest['max_voyage_time'] >= Q3_START)
        | (df_manifest['ata_time'] >= Q3_START)
    )
    & (df_manifest['max_voyage_time'] < Q4_START)
    & (df_manifest['ata_time'] < Q4_START)
)
df_manifest.loc[m_cx_23, 'quarter_category'] = 'CROSSING_Q2_Q3_EXCLUDED'

m_cx_34 = (
    (df_manifest['quarter_category'] == 'OTHER_DROPPED')
    & (df_manifest['min_voyage_time'] < Q4_START)
    & (
        (df_manifest['max_voyage_time'] >= Q4_START)
        | (df_manifest['ata_time'] >= Q4_START)
    )
)
df_manifest.loc[m_cx_34, 'quarter_category'] = 'CROSSING_Q3_Q4_EXCLUDED'

m_cx_other = df_manifest['quarter_category'] == 'OTHER_DROPPED'
df_manifest.loc[
    m_cx_other, 'quarter_category'
] = 'CROSSING_MULTI_OR_BOUNDARY_EXCLUDED'

df_manifest.to_parquet(
    MANIFEST_QUARTER_PARQUET, index=False, compression=PARQUET_COMPRESSION
)

Q1_TRIPS: Set[str] = set(
    df_manifest.loc[df_manifest['quarter_category'] == 'Q1_PURE', 'trip_id']
)
Q2_TRIPS: Set[str] = set(
    df_manifest.loc[df_manifest['quarter_category'] == 'Q2_PURE', 'trip_id']
)
Q3_TRIPS: Set[str] = set(
    df_manifest.loc[df_manifest['quarter_category'] == 'Q3_PURE', 'trip_id']
)
Q4_TRIPS: Set[str] = set(
    df_manifest.loc[df_manifest['quarter_category'] == 'Q4_PURE', 'trip_id']
)

# KIỂM ĐỊNH TOÁN HỌC GIAO TẬP BẰNG 0 TUYỆT ĐỐI
assert len(Q1_TRIPS & Q2_TRIPS) == 0, 'Rò rỉ giao điểm Q1 & Q2!'
assert len(Q1_TRIPS & Q3_TRIPS) == 0, 'Rò rỉ giao điểm Q1 & Q3!'
assert len(Q1_TRIPS & Q4_TRIPS) == 0, 'Rò rỉ giao điểm Q1 & Q4!'
assert len(Q2_TRIPS & Q3_TRIPS) == 0, 'Rò rỉ giao điểm Q2 & Q3!'
assert len(Q2_TRIPS & Q4_TRIPS) == 0, 'Rò rỉ giao điểm Q2 & Q4!'
assert len(Q3_TRIPS & Q4_TRIPS) == 0, 'Rò rỉ giao điểm Q3 & Q4!'

TRIP_TO_Q: Dict[str, int] = {}
for tid in Q1_TRIPS:
    TRIP_TO_Q[tid] = 1
for tid in Q2_TRIPS:
    TRIP_TO_Q[tid] = 2
for tid in Q3_TRIPS:
    TRIP_TO_Q[tid] = 3
for tid in Q4_TRIPS:
    TRIP_TO_Q[tid] = 4

print('\n--- PHÂN BỔ HÀNH TRÌNH 4 QUÝ ---')
for cat_n, cnt in df_manifest['quarter_category'].value_counts().items():
    print(f'  * {cat_n:38s}: {cnt:7,d} trips')
print(
    f'[✓] Giao điểm hành trình giữa 4 quý = 0 ({time.time() - t_step1:.1f}s)'
)
del df_manifest
gc.collect()


### BƯỚC 2: FIT THAM SỐ ĐIỀN KHUYẾT ĐỘC LẬP 100% TRÊN QUÝ 1 (`Q1_PURE`)

Để đảm bảo tuyệt đối không có bất kỳ thông tin nào từ Quý 2, Quý 3, hay Quý 4 rò rỉ vào bộ tham số tiền xử lý:
- Chỉ quét các bản ghi thuộc `Q1_PURE` trong dữ liệu thô 18 cột.
- Qua khử trùng lặp và bộ lọc sạch nhãn `Q1` (`Target_Hours > 0` và $v_{\text{implied}} \ge 1.62\text{ knots}$).
- Tính trung vị có trọng số 3 cấp (`mmsi -> vessel_type -> Cargo/Tanker -> global`) cho `length`, `width`, `draft` và `status_mode`.
- Lưu thành tệp JSON (`q1_imputation_artifacts.json`, `train_imputation_artifacts.json`) và bảng tra cứu theo tàu (`q1_mmsi_medians.parquet`).


In [ ]:
"""Cell 8: Bước 2 - Fit tham số điền khuyết 100% trên duy nhất Q1_PURE."""

t_step2 = time.time()
print('=' * 75)
print('BƯỚC 2: FIT THAM SỐ ĐIỀN KHUYẾT ĐỘC LẬP 100% TRÊN QUÝ 1 (Q1_PURE)')
print('=' * 75)

fit_cols = [
    'mmsi',
    'trip_id',
    'base_date_time',
    'longitude',
    'latitude',
    'sog',
    'status',
    'vessel_type',
    'length',
    'width',
    'draft',
    'berth_lon',
    'berth_lat',
    'ata_time',
    'Target_Hours',
]

q1_fit_dedup = SlidingBatchDeduplicator(keep_days=7)
freq_tables: Dict[str, List[pd.DataFrame]] = {
    'length': [],
    'width': [],
    'draft': [],
}
status_counts: Dict[float, int] = {}
total_q1_clean_rows = 0
scanned_fit = 0
total_raw_rows = get_total_raw_18_rows()

for b_idx, df_b in enumerate(
    iter_raw_18_batches(columns=fit_cols, batch_size=BATCH_SIZE_SPLIT)
):
    scanned_fit += len(df_b)
    bdt = to_naive_datetime(df_b['base_date_time'])

    # CHỈ LẤY CÁC DÒNG THUỘC Q1_PURE TRONG RANH GIỚI QUÝ 1
    m_q1 = (
        df_b['trip_id'].isin(Q1_TRIPS)
        & (bdt >= Q1_START)
        & (bdt < Q2_START)
    )
    if not m_q1.any():
        continue

    df_q1_raw = df_b.loc[m_q1].copy()
    df_q1_raw['base_date_time'] = bdt.loc[m_q1]

    df_q1_u = q1_fit_dedup.filter_unique(df_q1_raw)
    df_q1_clean = apply_maritime_base_filters(
        df_q1_u, atd_prev_map=atd_prev_map, is_train=True
    )
    if len(df_q1_clean) == 0:
        continue

    total_q1_clean_rows += len(df_q1_clean)

    st_valid = pd.to_numeric(
        df_q1_clean['status'], errors='coerce'
    ).dropna()
    if len(st_valid) > 0:
        for k_st, v_cnt in st_valid.value_counts().items():
            k_f = float(k_st)
            status_counts[k_f] = status_counts.get(k_f, 0) + int(v_cnt)

    for col in ['length', 'width', 'draft']:
        vals = pd.to_numeric(df_q1_clean[col], errors='coerce')
        if col == 'draft':
            ok_m = vals.notna() & (vals > 0.0) & (vals <= 25.5)
        else:
            ok_m = vals.notna() & (vals > 0.0)

        if ok_m.any():
            sub_c = pd.DataFrame({
                'mmsi': df_q1_clean.loc[ok_m, 'mmsi'].to_numpy(
                    dtype=np.int64
                ),
                'vessel_type': df_q1_clean.loc[ok_m, 'vessel_type'].to_numpy(
                    dtype=np.int32
                ),
                'is_tanker': df_q1_clean.loc[ok_m, 'Is_Tanker'].to_numpy(
                    dtype=np.int32
                ),
                'val': vals.loc[ok_m].to_numpy(dtype=np.float32),
            })
            agg_c = (
                sub_c.groupby(
                    ['mmsi', 'vessel_type', 'is_tanker', 'val'],
                    as_index=False,
                )
                .size()
                .rename(columns={'size': 'cnt'})
            )
            freq_tables[col].append(agg_c)

    if (b_idx + 1) % 25 == 0:
        for col in ['length', 'width', 'draft']:
            if len(freq_tables[col]) > 1:
                merged_f = (
                    pd.concat(freq_tables[col], ignore_index=True)
                    .groupby(
                        ['mmsi', 'vessel_type', 'is_tanker', 'val'],
                        as_index=False,
                    )['cnt']
                    .sum()
                )
                freq_tables[col] = [merged_f]
        gc.collect()

    if (b_idx + 1) % 60 == 0 or scanned_fit >= total_raw_rows:
        print(
            f'  -> Quét Fit Q1: {scanned_fit:,}/{total_raw_rows:,} | '
            f'Q1 sạch tích lũy: {total_q1_clean_rows:,}'
        )

del q1_fit_dedup
gc.collect()

imputation_artifacts = {
    'fitted_on': 'Q1_PURE_CLEAN',
    'train_clean_rows': int(total_q1_clean_rows),
    'status_mode': (
        float(max(status_counts, key=status_counts.get))
        if status_counts
        else 0.0
    ),
    'global_medians': {},
    'group_medians': {'cargo': {}, 'tanker': {}},
    'type_medians': {'length': {}, 'width': {}, 'draft': {}},
    'mmsi_medians': {'length': {}, 'width': {}, 'draft': {}},
}

mmsi_records: Dict[int, Dict[str, float]] = {}

for col in ['length', 'width', 'draft']:
    if not freq_tables[col]:
        raise RuntimeError(f'Không có dữ liệu hợp lệ cho {col} ở Q1_PURE!')

    df_freq = (
        pd.concat(freq_tables[col], ignore_index=True)
        .groupby(
            ['mmsi', 'vessel_type', 'is_tanker', 'val'],
            as_index=False,
        )['cnt']
        .sum()
    )
    freq_tables[col] = []

    g_agg = df_freq.groupby('val', as_index=False)['cnt'].sum()
    g_med = compute_weighted_median(
        g_agg['val'].to_numpy(dtype=np.float64),
        g_agg['cnt'].to_numpy(dtype=np.int64),
    )
    imputation_artifacts['global_medians'][col] = round(g_med, 4)

    for g_code, g_name in [(0, 'cargo'), (1, 'tanker')]:
        sub_g = df_freq[df_freq['is_tanker'] == g_code]
        if len(sub_g) > 0:
            sub_agg = sub_g.groupby('val', as_index=False)['cnt'].sum()
            grp_med = compute_weighted_median(
                sub_agg['val'].to_numpy(dtype=np.float64),
                sub_agg['cnt'].to_numpy(dtype=np.int64),
            )
        else:
            grp_med = g_med
        imputation_artifacts['group_medians'][g_name][col] = round(
            grp_med, 4
        )

    sub_vt = df_freq.groupby(['vessel_type', 'val'], as_index=False)[
        'cnt'
    ].sum()
    for vt_val, grp_vt in sub_vt.groupby('vessel_type'):
        vt_med = compute_weighted_median(
            grp_vt['val'].to_numpy(dtype=np.float64),
            grp_vt['cnt'].to_numpy(dtype=np.int64),
        )
        imputation_artifacts['type_medians'][col][int(vt_val)] = round(
            vt_med, 4
        )

    sub_mm = df_freq.groupby(['mmsi', 'val'], as_index=False)['cnt'].sum()
    for mm_val, grp_m in sub_mm.groupby('mmsi'):
        mm_int = int(mm_val)
        if len(grp_m) == 1:
            m_med = float(grp_m['val'].iloc[0])
        else:
            m_med = compute_weighted_median(
                grp_m['val'].to_numpy(dtype=np.float64),
                grp_m['cnt'].to_numpy(dtype=np.int64),
            )
        r_val = round(m_med, 4)
        imputation_artifacts['mmsi_medians'][col][mm_int] = r_val
        mmsi_records.setdefault(mm_int, {})[col] = r_val

    del df_freq, sub_vt, sub_mm
    gc.collect()

# Lưu bảng mmsi_medians của Q1 ra Parquet gọn nhẹ để tái sử dụng
mmsi_rows = [
    {
        'mmsi': k,
        'length': v.get('length', np.nan),
        'width': v.get('width', np.nan),
        'draft': v.get('draft', np.nan),
    }
    for k, v in mmsi_records.items()
]
pd.DataFrame(mmsi_rows).to_parquet(
    Q1_MMSI_MEDIANS_PARQUET, index=False, compression=PARQUET_COMPRESSION
)

json_summary = {
    'fitted_on': imputation_artifacts['fitted_on'],
    'train_clean_rows': imputation_artifacts['train_clean_rows'],
    'status_mode': imputation_artifacts['status_mode'],
    'global_medians': imputation_artifacts['global_medians'],
    'group_medians': imputation_artifacts['group_medians'],
    'type_medians': {
        c: {
            str(k): v
            for k, v in imputation_artifacts['type_medians'][c].items()
        }
        for c in ['length', 'width', 'draft']
    },
    'mmsi_lookup_count': {
        c: len(imputation_artifacts['mmsi_medians'][c])
        for c in ['length', 'width', 'draft']
    },
}

for j_path in [IMPUTE_ARTIFACTS_JSON, Q1_ARTIFACTS_JSON]:
    with open(j_path, 'w', encoding='utf-8') as f_out:
        json.dump(json_summary, f_out, ensure_ascii=False, indent=2)

print(
    f'[✓] Hoàn tất Fit Q1_PURE từ {total_q1_clean_rows:,} dòng trong '
    f'{time.time() - t_step2:.1f}s'
)
print(f'    • status_mode    : {imputation_artifacts["status_mode"]}')
print(f'    • global_medians : {imputation_artifacts["global_medians"]}')
print(f'    • group_medians  : {imputation_artifacts["group_medians"]}')


### BƯỚC 3: STREAMING XUẤT 12 TỆP PARQUET 4 QUÝ CHO 3 PHIÊN BẢN (CHỈ TIỀN XỬ LÝ SẴN Q1)

Theo **Phương án 2 (Zero Leakage)**:
1. **`01_ban_tho_18_thuoc_tinh/q1..q4.parquet` (18 cột):** Bảo toàn nguyên vẹn dữ liệu thô gắn nhãn của từng quý thuần túy.
2. **`02_ban_nen_tang_22_thuoc_tinh/q1..q4.parquet` (22 cột) & `03_ban_tien_xu_ly_27_thuoc_tinh/q1..q4.parquet` (27 cột):**
   - **Quý 1 (`q1.parquet`):** Áp dụng khử trùng lặp -> lọc nhãn (`Target_Hours > 0` & $v_{\text{implied}} \ge 1.62\text{ kn}$) -> **điền khuyết đầy đủ bằng tham số `Q1_PURE`** (`apply_q1_fitted_transform`).
   - **Quý 2, Quý 3, Quý 4 (`q2.parquet`, `q3.parquet`, `q4.parquet`):** Áp dụng khử trùng lặp độc lập theo quý -> lọc hàng hải (`is_train=False`, **không lọc theo `Target_Hours`**) -> tính đặc trưng hình học & lượng giác nhưng **GIỮ NGUYÊN `NaN` chưa điền khuyết** (`build_unimputed_features_for_eval_quarter`), đánh dấu `imputation_flag` $\in \{0, -1\}$ để tiền xử lý động trong từng kịch bản thực nghiệm.


In [ ]:
"""Cell 10: Bước 3 - Streaming xuất 12 tệp q1..q4.parquet cho 3 phiên bản."""

t_step3 = time.time()
print('=' * 75)
print('BƯỚC 3: XUẤT 12 TỆP 4 QUÝ (Q1 TIỀN XỬ LÝ SẴN | Q2..Q4 GIỮ NGUYÊN NaN)')
print('=' * 75)

q_keys = ['Q1', 'Q2', 'Q3', 'Q4']
q_bounds = {
    1: (Q1_START, Q2_START),
    2: (Q2_START, Q3_START),
    3: (Q3_START, Q4_START),
    4: (Q4_START, Q4_END),
}

writers_raw: Dict[int, pq.ParquetWriter] = {}
writers_base: Dict[int, pq.ParquetWriter] = {}
writers_prep: Dict[int, pq.ParquetWriter] = {}
dedup_by_q: Dict[int, SlidingBatchDeduplicator] = {}

for q_name in q_keys:
    q_code = QUARTER_SPECS[q_name]['code']
    f_name = QUARTER_SPECS[q_name]['file']
    writers_raw[q_code] = pq.ParquetWriter(
        os.path.join(DIR_RAW_SPLIT, f_name),
        SCHEMA_RAW_LABELED,
        compression=PARQUET_COMPRESSION,
    )
    writers_base[q_code] = pq.ParquetWriter(
        os.path.join(DIR_BASE_SPLIT, f_name),
        SCHEMA_BASE,
        compression=PARQUET_COMPRESSION,
    )
    writers_prep[q_code] = pq.ParquetWriter(
        os.path.join(DIR_PREP_SPLIT, f_name),
        SCHEMA_PREPROCESSED,
        compression=PARQUET_COMPRESSION,
    )
    dedup_by_q[q_code] = SlidingBatchDeduplicator(keep_days=7)

row_counts = {
    q: {'raw': 0, 'base': 0, 'prep': 0} for q in [1, 2, 3, 4]
}
dropped_crossing_rows = 0
scanned_s3 = 0
total_raw_rows = get_total_raw_18_rows()

try:
    for b_idx, df_b in enumerate(
        iter_raw_18_batches(columns=COLS_RAW, batch_size=BATCH_SIZE_SPLIT)
    ):
        scanned_s3 += len(df_b)
        bdt = to_naive_datetime(df_b['base_date_time'])
        ata_n = to_naive_datetime(df_b['ata_time'])
        q_codes = df_b['trip_id'].map(TRIP_TO_Q).fillna(0).astype(int)

        matched_in_batch = 0
        for q_code in [1, 2, 3, 4]:
            t_start, t_end = q_bounds[q_code]
            m_q = (
                (q_codes == q_code)
                & (bdt >= t_start)
                & (bdt < t_end)
            )
            if not m_q.any():
                continue

            sub_raw = df_b.loc[m_q].copy()
            sub_raw['trip_id'] = sub_raw['trip_id'].astype(str)
            sub_raw['base_date_time'] = bdt.loc[m_q]
            sub_raw['ata_time'] = ata_n.loc[m_q]
            matched_in_batch += len(sub_raw)

            # 1. Ghi Bản Thô 18 cột
            rb_raw = pa.RecordBatch.from_pandas(
                sub_raw[COLS_RAW],
                schema=SCHEMA_RAW_LABELED,
                preserve_index=False,
            )
            writers_raw[q_code].write_batch(rb_raw)
            row_counts[q_code]['raw'] += len(sub_raw)

            # 2. Khử trùng lặp & Lọc hàng hải độc lập theo quý
            sub_u = dedup_by_q[q_code].filter_unique(sub_raw)
            is_q1 = q_code == 1
            sub_clean = apply_maritime_base_filters(
                sub_u, atd_prev_map=atd_prev_map, is_train=is_q1
            )
            if len(sub_clean) > 0:
                if is_q1:
                    # CHỈ Q1 được điền khuyết sẵn từ tham số fit trên Q1
                    df_22, df_27 = apply_q1_fitted_transform(
                        sub_clean, imputation_artifacts
                    )
                else:
                    # Q2, Q3, Q4 giữ nguyên NaN chưa điền khuyết
                    df_22, df_27 = build_unimputed_features_for_eval_quarter(
                        sub_clean
                    )

                writers_base[q_code].write_batch(
                    pa.RecordBatch.from_pandas(
                        df_22, schema=SCHEMA_BASE, preserve_index=False
                    )
                )
                writers_prep[q_code].write_batch(
                    pa.RecordBatch.from_pandas(
                        df_27,
                        schema=SCHEMA_PREPROCESSED,
                        preserve_index=False,
                    )
                )
                row_counts[q_code]['base'] += len(df_22)
                row_counts[q_code]['prep'] += len(df_27)
                del df_22, df_27

            del sub_raw, sub_u, sub_clean

        dropped_crossing_rows += len(df_b) - matched_in_batch
        del df_b

        if (b_idx + 1) % 25 == 0 or scanned_s3 >= total_raw_rows:
            pct = (
                (scanned_s3 / total_raw_rows) * 100.0
                if total_raw_rows > 0
                else 100.0
            )
            print(
                f'  -> [{scanned_s3:,}/{total_raw_rows:,} ({pct:5.1f}%)] '
                f'Prep Q1={row_counts[1]["prep"]:,} | '
                f'Q2={row_counts[2]["prep"]:,} | '
                f'Q3={row_counts[3]["prep"]:,} | '
                f'Q4={row_counts[4]["prep"]:,} | '
                f'Dropped={dropped_crossing_rows:,}'
            )
            gc.collect()
finally:
    for q_code in [1, 2, 3, 4]:
        writers_raw[q_code].close()
        writers_base[q_code].close()
        writers_prep[q_code].close()

del dedup_by_q
gc.collect()

# Nếu raw_labeled.parquet nằm trong BASE_WORK_DIR do cold-start tạo ra,
# có thể xóa bớt sau khi đã tách xong 4 quý để giữ đĩa working < 19.5 GiB.
if (
    CLEANUP_TEMP_AFTER_MERGE
    and os.path.isfile(RAW_LABELED_WORK_PARQUET)
    and os.path.abspath(RAW_LABELED_WORK_PARQUET).startswith(
        os.path.abspath(BASE_WORK_DIR)
    )
):
    os.remove(RAW_LABELED_WORK_PARQUET)
    print(f'[INFO] Đã dọn {RAW_LABELED_WORK_PARQUET} giải phóng đĩa.')

print(f'[✓] Hoàn tất Bước 3 trong {time.time() - t_step3:.1f}s.')


In [ ]:
"""Cell 11: Bước 4 - Kiểm toán toàn diện 12 tệp, Overlap = 0 & NaN Audit."""

print('=' * 75)
print('BƯỚC 4: KIỂM TOÁN 12 TỆP PARQUET 4 QUÝ & CHỐNG DATA LEAKAGE')
print('=' * 75)

version_dirs = {
    '01_ban_tho_18_thuoc_tinh': (DIR_RAW_SPLIT, 18, False),
    '02_ban_nen_tang_22_thuoc_tinh': (DIR_BASE_SPLIT, 22, False),
    '03_ban_tien_xu_ly_27_thuoc_tinh': (DIR_PREP_SPLIT, 27, True),
}

inv_rows = []
prep_trips_by_q: Dict[str, Set[str]] = {}

for v_name, (v_dir, exp_cols, is_prep) in version_dirs.items():
    for q_name, q_info in QUARTER_SPECS.items():
        f_name = q_info['file']
        fpath = os.path.join(v_dir, f_name)
        assert os.path.isfile(fpath), f'Thiếu tệp {fpath}!'

        pf = pq.ParquetFile(fpath)
        assert pf.metadata.num_columns == exp_cols, (
            f'Sai số cột {v_name}/{f_name}: '
            f'{pf.metadata.num_columns} != {exp_cols}'
        )

        sz_mb = round(os.path.getsize(fpath) / (1024 * 1024), 2)
        q_s = pd.Timestamp(q_info['start'])
        q_e = pd.Timestamp(q_info['end'])

        read_cols = [
            'trip_id',
            'base_date_time',
            'ata_time',
            'length',
            'width',
            'draft',
            'cog',
            'heading',
            'status',
            'Target_Hours',
        ]
        if is_prep:
            read_cols.append('imputation_flag')

        trips_q: Set[str] = set()
        min_dt = None
        max_dt = None
        min_ata = None
        max_ata = None
        nan_cols_chk = ['length', 'width', 'draft', 'cog', 'heading']
        nan_cnts = {c: 0 for c in nan_cols_chk}
        flag_cnts: Dict[int, int] = {}
        n_rows = 0

        for batch in pf.iter_batches(batch_size=1_500_000, columns=read_cols):
            df_c = batch.to_pandas()
            n_rows += len(df_c)
            trips_q.update(df_c['trip_id'].unique().tolist())

            b_min = df_c['base_date_time'].min()
            b_max = df_c['base_date_time'].max()
            a_min = df_c['ata_time'].min()
            a_max = df_c['ata_time'].max()
            min_dt = b_min if min_dt is None else min(min_dt, b_min)
            max_dt = b_max if max_dt is None else max(max_dt, b_max)
            min_ata = a_min if min_ata is None else min(min_ata, a_min)
            max_ata = a_max if max_ata is None else max(max_ata, a_max)

            for c in nan_cnts:
                nan_cnts[c] += int(df_c[c].isna().sum())
            if is_prep:
                for fk, fv in df_c['imputation_flag'].value_counts().items():
                    flag_cnts[int(fk)] = flag_cnts.get(int(fk), 0) + int(fv)

        if n_rows > 0:
            assert min_dt >= q_s and max_dt < q_e, (
                f'Vi phạm mốc base_date_time {v_name}/{q_name}!'
            )
            assert min_ata >= q_s and max_ata < q_e, (
                f'Vi phạm mốc ata_time {v_name}/{q_name}!'
            )

        # Kiểm chứng nguyên tắc Phương án 2 trên Bản 22 & 27 cột
        if v_name != '01_ban_tho_18_thuoc_tinh' and n_rows > 0:
            if q_name == 'Q1':
                for c in ['length', 'width', 'draft', 'cog', 'heading']:
                    assert nan_cnts[c] == 0, (
                        f'{v_name} Q1 còn NaN ở cột {c}!'
                    )
            elif is_prep:
                assert all(fk in (0, -1) for fk in flag_cnts), (
                    f'{v_name} {q_name} không được điền khuyết sẵn!'
                )

        if is_prep:
            prep_trips_by_q[q_name] = trips_q

        nan_len_pct = (
            round(nan_cnts['length'] / n_rows * 100.0, 2) if n_rows else 0.0
        )
        nan_drf_pct = (
            round(nan_cnts['draft'] / n_rows * 100.0, 2) if n_rows else 0.0
        )
        inv_rows.append({
            'Version': v_name,
            'Quarter': q_name,
            'File_Name': f_name,
            'Num_Rows': n_rows,
            'Num_Cols': exp_cols,
            'Size_MB': sz_mb,
            'Unique_Trips': len(trips_q),
            'NaN_Len_%': nan_len_pct,
            'NaN_Draft_%': nan_drf_pct,
            'Min_Date': str(min_dt),
            'Max_Date': str(max_dt),
        })
        if is_prep:
            print(
                f'  [{q_name}] rows={n_rows:,} | flag_counts={flag_cnts} | '
                f'NaN length={nan_len_pct}% | draft={nan_drf_pct}%'
            )

df_inv = pd.DataFrame(inv_rows)
print('\n' + df_inv.to_string(index=False))
inv_csv_path = os.path.join(BASE_WORK_DIR, 'quarter_dataset_inventory.csv')
df_inv.to_csv(inv_csv_path, index=False)
print(f'\n[✓] Đã xuất bảng kiểm kê: {inv_csv_path}')

# Ma trận giao điểm 4x4 trên Bản 27 cột
q_list = ['Q1', 'Q2', 'Q3', 'Q4']
ov_mat = np.zeros((4, 4), dtype=int)
for i, qi in enumerate(q_list):
    for j, qj in enumerate(q_list):
        ov_mat[i, j] = len(prep_trips_by_q[qi] & prep_trips_by_q[qj])
        if i != j:
            assert ov_mat[i, j] == 0, f'Lỗi trùng chuyến {qi} vs {qj}!'

print('\n--- MA TRẬN GIAO ĐIỂM HÀNH TRÌNH (TRIP OVERLAP = 0) ---')
print(pd.DataFrame(ov_mat, index=q_list, columns=q_list).to_string())

d_tot, d_usd, d_fre = shutil.disk_usage(BASE_WORK_DIR)
print('-' * 75)
print(
    f'[ĐĨA WORKING]: Đã dùng {d_usd / (1024 ** 3):.2f} GiB / '
    f'{d_tot / (1024 ** 3):.2f} GiB '
    f'(Còn trống {d_fre / (1024 ** 3):.2f} GiB)'
)
print('>>> PIPELINE 4 QUÝ (PRE-SPLIT & CHỈ PREPROCESS Q1) ĐÃ HOÀN TẤT! <<<')
